## Analyze runs (Colab / local)

Loads `metrics.json` files under `runs/` and produces a quick table.


In [ ]:
import json
from pathlib import Path

BENCHMARK_ROOT = Path('/content/nav4rails/repositories/FineTuningOnTelecomCluster/benchmark').resolve()
runs_root = BENCHMARK_ROOT / 'runs'

rows = []
for run_dir in runs_root.rglob('*_exp*'):
    metrics_path = run_dir / 'metrics.json'
    if metrics_path.is_file():
        rows.append(json.loads(metrics_path.read_text(encoding='utf-8')))

print('runs:', len(rows))
rows[:2]

In [ ]:
# Simple markdown table without pandas

def md_table(rows, cols):
    out = []
    out.append('| ' + ' | '.join(cols) + ' |')
    out.append('| ' + ' | '.join(['---']*len(cols)) + ' |')
    for r in rows:
        out.append('| ' + ' | '.join(str(r.get(c,'')) for c in cols) + ' |')
    return '\n'.join(out)

cols = ['run_id','method','model_name','xml_valid','latency_ms','vram_mb']
print(md_table(rows[:20], cols))

In [ ]:
# Aggregate L1/L2/L3 pass rates from validation_report.json

import json
from pathlib import Path

BENCHMARK_ROOT = Path('/content/nav4rails/repositories/FineTuningOnTelecomCluster/benchmark').resolve()
runs_root = BENCHMARK_ROOT / 'runs'

reports = []
for run_dir in runs_root.rglob('*_exp*'):
    vr = run_dir / 'validation_report.json'
    mt = run_dir / 'metrics.json'
    if vr.is_file() and mt.is_file():
        rep = json.loads(vr.read_text(encoding='utf-8'))
        met = json.loads(mt.read_text(encoding='utf-8'))
        # Determine per-layer pass (no errors in that layer)
        by_layer = (rep.get('summary') or {}).get('by_layer') or {}
        def layer_ok(layer):
            return (by_layer.get(layer) or {}).get('errors', 0) == 0
        reports.append({
            'run_id': met.get('run_id', run_dir.name),
            'method': met.get('method'),
            'model': met.get('model'),
            'xml_valid': met.get('xml_valid'),
            'pass_l1': layer_ok('L1'),
            'pass_l2': layer_ok('L2'),
            'pass_l3': layer_ok('L3'),
            'errors': (rep.get('summary') or {}).get('errors'),
        })

print('runs_with_validation:', len(reports))

# Simple aggregation
from collections import defaultdict
agg = defaultdict(lambda: {'n':0,'xml_valid':0,'pass_l1':0,'pass_l2':0,'pass_l3':0})
for r in reports:
    key = str(r.get('method'))
    a = agg[key]
    a['n'] += 1
    a['xml_valid'] += 1 if r.get('xml_valid') else 0
    a['pass_l1'] += 1 if r.get('pass_l1') else 0
    a['pass_l2'] += 1 if r.get('pass_l2') else 0
    a['pass_l3'] += 1 if r.get('pass_l3') else 0

rows = []
for method, a in sorted(agg.items()):
    n = max(1, a['n'])
    rows.append({
        'method': method,
        'n': a['n'],
        'xml_valid_rate': round(a['xml_valid']/n, 3),
        'pass_l1_rate': round(a['pass_l1']/n, 3),
        'pass_l2_rate': round(a['pass_l2']/n, 3),
        'pass_l3_rate': round(a['pass_l3']/n, 3),
    })

# Print markdown table
cols = ['method','n','xml_valid_rate','pass_l1_rate','pass_l2_rate','pass_l3_rate']
print('| ' + ' | '.join(cols) + ' |')
print('| ' + ' | '.join(['---']*len(cols)) + ' |')
for r in rows:
    print('| ' + ' | '.join(str(r[c]) for c in cols) + ' |')
